In [1]:
print("ok")

ok


In [ ]:
%pwd

'c:\\Users\\Ankush\\project\\ai_diagonisis_project_bai\\AI_Diagnosis_project-\\research'

In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
# Extract text from PDF files
def load_pdf_files(Data):
    loader = DirectoryLoader(
        Data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [3]:


extracted_docs = load_pdf_files("C:/Users/Ankush/project/ai_diagonisis_project_bai/AI_Diagnosis_project-/Data")



In [15]:
# Split the data into text chunks 
def text_split(extracted_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_docs)
    return text_chunks

In [18]:
text_chunks = text_split(extracted_docs)
print("length of text chunks :", len(text_chunks))

length of text chunks : 42950


In [4]:
from langchain.embeddings import HuggingFaceEmbeddings



In [6]:

def download_hugging_face_embeddings():
    
    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embeddings


In [7]:
embeddings = download_hugging_face_embeddings()

C:\Users\Ankush\AppData\Local\Temp\ipykernel_20388\1466506747.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
c:\Users\Ankush\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
query_result = embeddings.embed_query("hello world")
print(len(query_result))

384


In [21]:


from dotenv import load_dotenv
import os
load_dotenv()



True

In [30]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

In [19]:
# from pinecone import Pinecone 
# pinecone_api_key = PINECONE_API_KEY

# pc = Pinecone(api_key=pinecone_api_key)

In [31]:
# from pinecone import Pinecone, ServerlessSpec

# pc = Pinecone(api_key="YOUR_API_KEY")

# index_name = "medical-chatbot"

# # Get list of existing indexes
# existing_indexes = [i.name for i in pc.list_indexes()]

# if index_name not in existing_indexes:
#     pc.create_index(
#         name=index_name,
#         dimension=384,
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1")
#     )

# index = pc.Index(index_name)

# print("Index ready ✅")

In [12]:
import os
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

# Check existing indexes
existing_indexes = [i.name for i in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created ✅")
else:
    print("Index already exists ✅")

# Connect to index
index = pc.Index(index_name)
print("Connected to index 🚀")

Index already exists ✅
Connected to index 🚀


In [13]:
print(pc.list_indexes())

{'indexes': [{'deletion_protection': 'disabled',
              'dimension': 384,
              'host': 'medicalbot-ixgnf6h.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'medicalbot',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}},
             {'deletion_protection': 'disabled',
              'dimension': 1024,
              'host': 'medical-chatbot-ixgnf6h.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'medical-chatbot',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}


In [19]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

KeyboardInterrupt: 

In [20]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [21]:
docsearch

In [23]:


retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
print(retriever)



tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x0000013D754673A0> search_kwargs={'k': 3}


In [26]:


retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs[0].page_content




'Acne— A chronic inflammation of the sebaceous\nglands that manifests as blackheads, whiteheads,and/or pustules on the face or trunk.\nPsoriasis— A skin disorder of chronic, itchy scaling\nmost commonly at sites of repeated minor trauma(e.g. elbows, knees, and skin folds). It affects up to2% of the population in Western countries—malesand females equally.\nRosacea— A chronic inflammation of the face, with'

In [35]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# print(OPENAI_API_KEY)  # DEBUG
os.environ["OPENAI_API_KEY"] =OPENAI_API_KEY

from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")



In [36]:


from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate



In [37]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [38]:

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

